In [ ]:
import os, time
from pathlib import Path
import torch
import psutil
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown

%matplotlib inline

display(Markdown("## Part 0 — Setup"))
display(Markdown("_Imports, helpers, RAM tracker, and the model ID. Nothing is downloaded or loaded yet._"))

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["font.size"] = 10

CATEGORY_COLORS = {
    "weights":          "#d62728",
    "tokenizer":        "#2ca02c",
    "model-config":     "#1f77b4",
    "metadata/license": "#9467bd",
    "other":            "#7f7f7f",
}

PROC = psutil.Process(os.getpid())
def ram_mb(): return PROC.memory_info().rss / (1024 ** 2)

def fmt_size(b):
    if b < 1024:        return f"{b} B"
    if b < 1024**2:     return f"{b/1024:.2f} KB"
    if b < 1024**3:     return f"{b/1024**2:.2f} MB"
    return f"{b/1024**3:.2f} GB"

STAGE_LOG = []

class Stage:
    """Reusable timing + RAM tracker. Use in every level."""
    def __init__(self, name, reads=""):
        self.name, self.reads = name, reads
    def __enter__(self):
        print(f"▶ {self.name}")
        if self.reads: print(f"  reads: {self.reads}")
        self.ram_start = ram_mb()
        self.t_start = time.time()
        return self
    def __exit__(self, *a):
        dt = time.time() - self.t_start
        ram_end = ram_mb()
        delta = ram_end - self.ram_start
        STAGE_LOG.append({
            "stage":         self.name,
            "reads":         self.reads,
            "ram_before_MB": round(self.ram_start, 1),
            "ram_after_MB":  round(ram_end, 1),
            "ram_delta_MB":  round(delta, 1),
            "time_s":        round(dt, 2),
        })
        print(f"  ✓ {dt:.2f}s   ΔRAM +{delta:.1f} MB   total {ram_end:.1f} MB\n")

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                PART 00: SETUP COMPLETE                   @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("========== ENVIRONMENT ==========")
print("model id   :", MODEL_ID)
print("torch ver  :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
print("baseline RAM (notebook only) :", f"{ram_mb():.1f} MB")
print("=================================")

In [ ]:
display(Markdown("## Part 1 — Download stage"))
display(Markdown("_Where do files land on disk? What is fetched and in what order?_"))

from huggingface_hub import snapshot_download, scan_cache_dir

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@               PART 01: DOWNLOAD STAGE                    @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

def cache_info(model_id):
    try:
        for r in scan_cache_dir().repos:
            if r.repo_id == model_id:
                return True, r.size_on_disk
    except Exception: pass
    return False, 0

cached_before, size_before = cache_info(MODEL_ID)
display(Markdown(f"**Cache before:** hit=`{cached_before}`" +
                 (f", size=`{fmt_size(size_before)}`" if cached_before else "")))

print("========== STEP 1: CACHE CHECK ==========")
print("cache hit :", cached_before)
if cached_before:
    print("cached size :", fmt_size(size_before))
print("=========================================")

print()
print("========== STEP 2: SNAPSHOT DOWNLOAD ==========")
with Stage("snapshot_download", reads="huggingface hub"):
    local_path = snapshot_download(repo_id=MODEL_ID)
print("local path :", local_path)
print("===============================================")

hf_root = Path(os.path.expanduser("~/.cache/huggingface/hub"))
display(Markdown(f"""
**Cache layout**
- root: `{hf_root}`
- snapshot: `{local_path}`
- relative: `{Path(local_path).relative_to(hf_root)}`
"""))

def categorize(name):
    n = name.lower()
    if n.endswith((".safetensors", ".bin")):                return "weights"
    if n in ("config.json", "generation_config.json"):      return "model-config"
    if "tokenizer" in n or n == "special_tokens_map.json":  return "tokenizer"
    if n.endswith((".md", ".txt")):                         return "metadata/license"
    return "other"

rows = []
for f in os.listdir(local_path):
    full = os.path.join(local_path, f)
    if os.path.isfile(full) or os.path.islink(full):
        s = os.path.getsize(full)
        rows.append({"filename": f, "category": categorize(f),
                     "size_bytes": s, "size": fmt_size(s)})

df_files = pd.DataFrame(rows).sort_values("size_bytes").reset_index(drop=True)
df_files.index = range(1, len(df_files) + 1)
df_files.index.name = "#"

display(Markdown("**Files on disk (small → large = fetch order)**"))
display(df_files[["filename", "category", "size"]])

print()
print("========== STEP 3: FILES ON DISK ==========")
print(f"file count : {len(df_files)}")
print(f"total size : {fmt_size(df_files['size_bytes'].sum())}")
for _, row in df_files.iterrows():
    print(f"  {row['category']:18s} {row['size']:>12s}  {row['filename']}")
print("===========================================")

# horizontal bar — file sizes (log scale)
fig, ax = plt.subplots(figsize=(10, max(3, len(df_files) * 0.4)))
ax.barh(df_files["filename"], df_files["size_bytes"],
        color=[CATEGORY_COLORS[c] for c in df_files["category"]])
ax.set_xscale("log")
ax.set_xlabel("size (bytes, log scale)")
ax.set_title("File sizes on disk")
ax.invert_yaxis()
cats_present = df_files["category"].unique()
ax.legend(handles=[Patch(facecolor=CATEGORY_COLORS[c], label=c) for c in cats_present],
          loc="lower right")
plt.tight_layout(); plt.show()

# # pie — disk usage by category
# cat_sizes = df_files.groupby("category")["size_bytes"].sum().sort_values(ascending=False)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.pie(cat_sizes.values, labels=cat_sizes.index,
#        colors=[CATEGORY_COLORS[c] for c in cat_sizes.index],
#        autopct=lambda p: f"{p:.1f}%\n({fmt_size(p * cat_sizes.sum() / 100)})",
#        startangle=90)
# ax.set_title("Disk usage by category"); plt.tight_layout(); plt.show()

total = df_files["size_bytes"].sum()
display(Markdown(f"""
**Summary** — files: `{len(df_files)}`, total: `{fmt_size(total)}`, fresh: `{not cached_before}`
"""))

In [ ]:
display(Markdown("## Part 2 — Load into RAM stage"))
display(Markdown("_Disk → RAM. What is read, in what order, how much memory is consumed?_"))

from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@              PART 02: LOAD INTO RAM STAGE                @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

ram_baseline = ram_mb()
display(Markdown(f"**Baseline RAM (before model load):** `{ram_baseline:.1f} MB`"))

print("========== STEP 1: BASELINE RAM ==========")
print(f"RAM before any load : {ram_baseline:.1f} MB")
print("==========================================")

log_start = len(STAGE_LOG)

print()
print("========== STEP 2: LOAD TOKENIZER ==========")
print("reads : tokenizer.json, tokenizer_config.json, special_tokens_map.json")
with Stage("load tokenizer", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("tokenizer class :", type(tokenizer).__name__)
print("vocab size      :", tokenizer.vocab_size)
print("============================================")

print()
print("========== STEP 3: LOAD MODEL CONFIG ==========")
print("reads : config.json")
with Stage("load model config", reads="config.json"):
    config = AutoConfig.from_pretrained(MODEL_ID)
print("config class :", type(config).__name__)
print("architecture :", config.architectures[0] if config.architectures else "n/a")
print("===============================================")

print()
print("========== STEP 4: LOAD MODEL WEIGHTS ==========")
print("reads : *.safetensors")
with Stage("load model weights", reads="*.safetensors"):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    model.eval()
print("model class : ", type(model).__name__)
print("model dtype : ", next(model.parameters()).dtype)
print("model device: ", next(model.parameters()).device)
print("================================================")

df_load = pd.DataFrame(STAGE_LOG[log_start:])
display(Markdown("**Step-by-step load log**"))
display(df_load)

# RAM growth bar
labels  = ["baseline"] + df_load["stage"].tolist()
values  = [ram_baseline] + df_load["ram_after_MB"].tolist()
deltas  = [0] + df_load["ram_delta_MB"].tolist()
colors  = ["#7f7f7f", "#2ca02c", "#1f77b4", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, values, color=colors)
for bar, d in zip(bars, deltas):
    h = bar.get_height()
    txt = f"{h:.0f} MB" + (f"\n(+{d:.0f})" if d > 0 else "")
    ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("RAM (MB)"); ax.set_title("RAM growth across load steps")
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout(); plt.show()

# time per step
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(df_load["stage"], df_load["time_s"], color=["#2ca02c", "#1f77b4", "#d62728"])
for i, v in enumerate(df_load["time_s"]):
    ax.text(v, i, f"  {v:.2f} s", va="center")
ax.set_xlabel("seconds"); ax.set_title("Time per load step")
plt.tight_layout(); plt.show()

total_params = sum(p.numel() for p in model.parameters())
param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())

print()
print("========== STEP 5: MODEL SUMMARY ==========")
print(f"architecture     : {config.architectures[0] if config.architectures else 'n/a'}")
print(f"hidden size      : {config.hidden_size}")
print(f"num layers       : {config.num_hidden_layers}")
print(f"vocab size       : {tokenizer.vocab_size}")
print(f"dtype            : {next(model.parameters()).dtype}")
print(f"device           : {next(model.parameters()).device}")
print(f"total parameters : {total_params:,}  ({total_params / 1e9:.3f} B)")
print(f"theoretical size : {fmt_size(param_bytes)}")
print(f"RAM by model     : {(ram_mb() - ram_baseline):.1f} MB")
print("===========================================")

display(Markdown(f"""
**Model summary**

| property | value |
|---|---|
| architecture     | `{config.architectures[0] if config.architectures else 'n/a'}` |
| hidden size      | `{config.hidden_size}` |
| num layers       | `{config.num_hidden_layers}` |
| vocab size       | `{tokenizer.vocab_size}` |
| dtype            | `{next(model.parameters()).dtype}` |
| device           | `{next(model.parameters()).device}` |
| total parameters | `{total_params:,}` ({total_params / 1e9:.3f} B) |
| theoretical size | `{fmt_size(param_bytes)}` |
| RAM by model     | `{(ram_mb() - ram_baseline):.1f} MB` |
"""))

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 0: opaque box)
# ─────────────────────────────────────────────────────────────

display(Markdown("## Part 3 — Inference (Level 0: opaque box)"))
display(Markdown("_Black box. Text in, text out. Pipeline middle is opaque at this level — it opens progressively from Level 1 onwards._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@          PART 03: INFERENCE (Level 0: opaque box)        @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── run inference ───────────────────────────────────────────
text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)

# ═════════════════════════════════════════════════════════════
# PLAIN-PRINT VIEW (Pattern B — terminal-friendly, always present)
# ═════════════════════════════════════════════════════════════

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                    STEP 1: LLM (opaque)                  @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

print("========== INPUT ==========")
print("text  :", repr(text))
print("shape :", input_ids.shape)
print("dtype :", input_ids.dtype)
print("ndim  :", input_ids.ndim)
print("===========================")
print()

print("========== MIDDLE (opaque at Level 0) ==========")
print("step  : model.generate(...)")
print("op    : opaque")
print("in    :", tuple(input_ids.shape), str(input_ids.dtype))
print("out   :", tuple(output_ids.shape), str(output_ids.dtype))
print("note  : internals hidden — opens progressively from Level 1 onwards")
print("================================================")
print()

print("========== OUTPUT ==========")
print("text  :", repr(output_text))
print("shape :", output_ids.shape)
print("dtype :", output_ids.dtype)
print("ndim  :", output_ids.ndim)
print("============================")

# ═════════════════════════════════════════════════════════════
# RICH-DISPLAY VIEW (Pattern A — DataFrames, grows each level)
# ═════════════════════════════════════════════════════════════

display(Markdown("---"))
display(Markdown("### Rich view"))

# INPUT / OUTPUT table
display(Markdown("**Input and Output tensors**"))
df_io = pd.DataFrame([
    {
        "side":  "INPUT",
        "text":  repr(text),
        "shape": str(tuple(input_ids.shape)),
        "dtype": str(input_ids.dtype),
        "ndim":  input_ids.ndim,
        "numel": input_ids.numel(),
    },
    {
        "side":  "OUTPUT",
        "text":  repr(output_text),
        "shape": str(tuple(output_ids.shape)),
        "dtype": str(output_ids.dtype),
        "ndim":  output_ids.ndim,
        "numel": output_ids.numel(),
    },
]).set_index("side")
display(df_io)

# PIPELINE table — at Level 0 it has 1 opaque row.
# At later levels we append rows (tokenize, embed, blocks, LM head, pick, decode).
display(Markdown("**Pipeline (middle) — at Level 0 this is one opaque step. Rows will grow each level.**"))
pipeline_rows = [
    {
        "step":           "model.generate",
        "op":             "opaque",
        "in_shape":       str(tuple(input_ids.shape)),
        "in_dtype":       str(input_ids.dtype),
        "out_shape":      str(tuple(output_ids.shape)),
        "out_dtype":      str(output_ids.dtype),
        "static_weights": "{model}  (vocab + all learned weights, bundled)",
        "dim_out":        "scalar (1 new token id)",
    },
]
df_pipeline = pd.DataFrame(pipeline_rows)
df_pipeline.index = range(1, len(df_pipeline) + 1)
df_pipeline.index.name = "#"
display(df_pipeline)